In [ ]:
# ============================================================
# FINBERT AS PRE-TRAINED NLP FEATURE EXTRACTOR
# For Sentiment Feature Generation Only
# ============================================================

!pip install transformers torch openpyxl -q

In [ ]:
# ============================================================
# 1. Import libraries
# ============================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

In [ ]:
# ============================================================
# 2. Load dataset
# ============================================================

file_path = "/content/FInal Data_Sentiment Analysis.xlsx"

df = pd.read_excel(file_path, sheet_name="Final Clean")

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace(" ", "_")
    .str.upper()
)

df = df.loc[:, ~df.columns.str.contains("^UNNAMED", case=False)]

print(df.shape)
display(df.head())

(857, 8)


,COMPANY_CODE,YEAR,QUARTER,HEADLINE,URL,SOURCE,CATEGORY,SUMMARY
0,3,2020,4,Second tranche bond issuance (Series C & D) la...,https://aboitiz.com/investor-relations/disclos...,Aboitiz IR,Debt Issuance,Issued Series C and D bonds as part of shelf r...
1,3,2020,4,Fixed-rate bonds issued under shelf program,https://s3-ap-southeast-1.amazonaws.com/aboiti...,Aboitiz Prospectus,Debt Refinancing,Issued peso bonds (Series C & D); proceeds use...
2,3,2020,4,Bond proceeds used for refinancing and water i...,https://cbonds.de/bonds/973289/,Cbonds,Debt Allocation,Proceeds used to refinance maturing bonds and ...
3,3,2021,1,Stable leverage maintained amid diversified po...,https://aboitiz.com/investor-relations/disclos...,AEV IR,Financial Position,AEV maintains stable debt profile supported by...
4,3,2021,1,Bond refinancing structure continues post-COVI...,https://aboitiz.com/investor-relations,disclosures,Aboitiz IR,Financial Position


In [ ]:
# ============================================================
# 3. Basic cleaning
# ============================================================

for col in ["HEADLINE", "SUMMARY", "URL", "SOURCE", "CATEGORY"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

df["COMPANY_CODE"] = df["COMPANY_CODE"].astype(int)
df["YEAR"] = df["YEAR"].astype(int)
df["QUARTER"] = df["QUARTER"].astype(int)

In [ ]:
# ============================================================
# 4. Remove duplicate articles
# ============================================================

df["DUPLICATE_KEY"] = (
    df["COMPANY_CODE"].astype(str) + "_" +
    df["YEAR"].astype(str) + "_" +
    df["QUARTER"].astype(str) + "_" +
    df["HEADLINE"].str.lower() + "_" +
    df["SUMMARY"].str.lower()
)

print("Duplicate records:", df["DUPLICATE_KEY"].duplicated().sum())

df = df.drop_duplicates(subset=["DUPLICATE_KEY"]).copy()
df = df.drop(columns=["DUPLICATE_KEY"])

print("Shape after duplicate removal:", df.shape)

Duplicate records: 0
Shape after duplicate removal: (857, 8)


In [ ]:
# ============================================================
# 5. Create text input for FinBERT
# ============================================================

df["TEXT_FOR_SENTIMENT"] = (
    df["HEADLINE"].astype(str).str.strip() + ". " +
    df["SUMMARY"].astype(str).str.strip()
)

df["TEXT_FOR_SENTIMENT"] = df["TEXT_FOR_SENTIMENT"].str.strip()

display(df[["COMPANY_CODE", "YEAR", "QUARTER", "TEXT_FOR_SENTIMENT"]].head())

,COMPANY_CODE,YEAR,QUARTER,TEXT_FOR_SENTIMENT
0,3,2020,4,Second tranche bond issuance (Series C & D) la...
1,3,2020,4,Fixed-rate bonds issued under shelf program. I...
2,3,2020,4,Bond proceeds used for refinancing and water i...
3,3,2021,1,Stable leverage maintained amid diversified po...
4,3,2021,1,Bond refinancing structure continues post-COVI...


In [ ]:
# ============================================================
# 6. Load FinBERT
# ============================================================

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
# ============================================================
# 7. FinBERT sentiment feature extraction function
# ============================================================

def extract_finbert_features(text):
    if pd.isna(text) or str(text).strip() == "":
        return pd.Series([np.nan, np.nan, np.nan, np.nan, "Missing"])

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    scores = outputs.logits.detach().numpy()[0]
    probabilities = softmax(scores)

    # ProsusAI/finbert label order:
    # 0 = positive, 1 = negative, 2 = neutral
    positive_prob = probabilities[0]
    negative_prob = probabilities[1]
    neutral_prob = probabilities[2]

    sentiment_score = positive_prob - negative_prob

    if sentiment_score > 0.10:
        sentiment_label = "Positive"
    elif sentiment_score < -0.10:
        sentiment_label = "Negative"
    else:
        sentiment_label = "Neutral"

    return pd.Series([
        positive_prob,
        neutral_prob,
        negative_prob,
        sentiment_score,
        sentiment_label
    ])

In [ ]:
# ============================================================
# 8. Apply FinBERT
# ============================================================

df[[
    "POSITIVE_PROB",
    "NEUTRAL_PROB",
    "NEGATIVE_PROB",
    "SENTIMENT_SCORE",
    "SENTIMENT_LABEL"
]] = df["TEXT_FOR_SENTIMENT"].apply(extract_finbert_features)

display(df[[
    "COMPANY_CODE",
    "YEAR",
    "QUARTER",
    "HEADLINE",
    "SENTIMENT_SCORE",
    "SENTIMENT_LABEL"
]].head(20))

,COMPANY_CODE,YEAR,QUARTER,HEADLINE,SENTIMENT_SCORE,SENTIMENT_LABEL
0,3,2020,4,Second tranche bond issuance (Series C & D) la...,0.049478,Neutral
1,3,2020,4,Fixed-rate bonds issued under shelf program,0.080950,Neutral
2,3,2020,4,Bond proceeds used for refinancing and water i...,0.097088,Neutral
3,3,2021,1,Stable leverage maintained amid diversified po...,0.935639,Positive
4,3,2021,1,Bond refinancing structure continues post-COVI...,0.344788,Positive
5,3,2021,2,Controlled borrowing supports infrastructure a...,0.255814,Positive
6,3,2021,2,Controlled borrowing supports group-wide capex...,0.020483,Neutral
7,3,2021,3,Series D bonds listed (multi-year bond program...,0.380804,Positive
8,3,2021,3,Bond refinancing cycle continues under fixed-r...,0.055296,Neutral
9,3,2021,4,Stable financial position supported by diversi...,0.920201,Positive


In [ ]:
# ============================================================
# 9. Check sentiment output
# ============================================================

display(df["SENTIMENT_LABEL"].value_counts())

display(df[[
    "POSITIVE_PROB",
    "NEUTRAL_PROB",
    "NEGATIVE_PROB",
    "SENTIMENT_SCORE"
]].describe())

,count
SENTIMENT_LABEL,
Positive,608
Neutral,172
Negative,77


,POSITIVE_PROB,NEUTRAL_PROB,NEGATIVE_PROB,SENTIMENT_SCORE
count,857.000000,857.000000,857.000000,857.000000
mean,0.512990,0.391658,0.095352,0.417638
std,0.377554,0.367859,0.238858,0.513692
min,0.006552,0.010245,0.006225,-0.965800
25%,0.100165,0.043622,0.010056,0.070553
50%,0.551217,0.215923,0.013671,0.502801
75%,0.913641,0.830626,0.023774,0.894486
max,0.960566,0.955836,0.975011,0.946021


In [ ]:
# ============================================================
# 10. Optional: Add category weights
# ============================================================

category_weights = {
    "Debt Default": 2.0,
    "Debt Restructuring": 2.0,
    "Credit Rating": 2.0,
    "Liquidity": 1.8,
    "Cash Flow": 1.5,
    "Financial Position": 1.5,
    "Interest Expense": 1.5,
    "Debt Position": 1.5,
    "Debt Issuance": 1.4,
    "Debt Refinancing": 1.4,
    "Capital Structure": 1.3,
    "Capital Raising": 1.3,
    "Investment Financing": 1.2,
    "Investment Expansion": 1.0
}

df["CATEGORY_WEIGHT"] = df["CATEGORY"].map(category_weights).fillna(1.0)

df["WEIGHTED_SENTIMENT_SCORE"] = (
    df["SENTIMENT_SCORE"] * df["CATEGORY_WEIGHT"]
)

display(df[[
    "CATEGORY",
    "CATEGORY_WEIGHT",
    "SENTIMENT_SCORE",
    "WEIGHTED_SENTIMENT_SCORE"
]].head())

,CATEGORY,CATEGORY_WEIGHT,SENTIMENT_SCORE,WEIGHTED_SENTIMENT_SCORE
0,Debt Issuance,1.4,0.049478,0.069270
1,Debt Refinancing,1.4,0.080950,0.113330
2,Debt Allocation,1.0,0.097088,0.097088
3,Financial Position,1.5,0.935639,1.403458
4,Aboitiz IR,1.0,0.344788,0.344788


In [ ]:
# ============================================================
# 11. Aggregate sentiment features by Company-Year-Quarter
# ============================================================

quarterly_sentiment = df.groupby(
    ["COMPANY_CODE", "YEAR", "QUARTER"]
).agg(
    AVG_SENTIMENT_SCORE=("SENTIMENT_SCORE", "mean"),
    MEDIAN_SENTIMENT_SCORE=("SENTIMENT_SCORE", "median"),
    MIN_SENTIMENT_SCORE=("SENTIMENT_SCORE", "min"),
    MAX_SENTIMENT_SCORE=("SENTIMENT_SCORE", "max"),
    SENTIMENT_VOLATILITY=("SENTIMENT_SCORE", "std"),

    AVG_WEIGHTED_SENTIMENT_SCORE=("WEIGHTED_SENTIMENT_SCORE", "mean"),
    MIN_WEIGHTED_SENTIMENT_SCORE=("WEIGHTED_SENTIMENT_SCORE", "min"),
    MAX_WEIGHTED_SENTIMENT_SCORE=("WEIGHTED_SENTIMENT_SCORE", "max"),

    AVG_POSITIVE_PROB=("POSITIVE_PROB", "mean"),
    AVG_NEUTRAL_PROB=("NEUTRAL_PROB", "mean"),
    AVG_NEGATIVE_PROB=("NEGATIVE_PROB", "mean"),

    NEWS_COUNT=("SENTIMENT_SCORE", "count"),
    POSITIVE_COUNT=("SENTIMENT_LABEL", lambda x: (x == "Positive").sum()),
    NEGATIVE_COUNT=("SENTIMENT_LABEL", lambda x: (x == "Negative").sum()),
    NEUTRAL_COUNT=("SENTIMENT_LABEL", lambda x: (x == "Neutral").sum())
).reset_index()

quarterly_sentiment["SENTIMENT_VOLATILITY"] = quarterly_sentiment["SENTIMENT_VOLATILITY"].fillna(0)

quarterly_sentiment["POSITIVE_RATIO"] = (
    quarterly_sentiment["POSITIVE_COUNT"] / quarterly_sentiment["NEWS_COUNT"]
)

quarterly_sentiment["NEGATIVE_RATIO"] = (
    quarterly_sentiment["NEGATIVE_COUNT"] / quarterly_sentiment["NEWS_COUNT"]
)

quarterly_sentiment["NEUTRAL_RATIO"] = (
    quarterly_sentiment["NEUTRAL_COUNT"] / quarterly_sentiment["NEWS_COUNT"]
)

display(quarterly_sentiment.head())

,COMPANY_CODE,YEAR,QUARTER,AVG_SENTIMENT_SCORE,MEDIAN_SENTIMENT_SCORE,MIN_SENTIMENT_SCORE,MAX_SENTIMENT_SCORE,SENTIMENT_VOLATILITY,AVG_WEIGHTED_SENTIMENT_SCORE,MIN_WEIGHTED_SENTIMENT_SCORE,...,AVG_POSITIVE_PROB,AVG_NEUTRAL_PROB,AVG_NEGATIVE_PROB,NEWS_COUNT,POSITIVE_COUNT,NEGATIVE_COUNT,NEUTRAL_COUNT,POSITIVE_RATIO,NEGATIVE_RATIO,NEUTRAL_RATIO
0,1,2020,4,0.046298,0.046298,0.009051,0.083544,0.052675,0.046298,0.009051,...,0.063455,0.919388,0.017157,2,0,0,2,0.0,0.0,1.0
1,1,2021,1,0.938165,0.938165,0.938165,0.938165,0.000000,1.407248,1.407248,...,0.953216,0.031733,0.015051,1,1,0,0,1.0,0.0,0.0
2,1,2021,2,0.106079,0.106079,0.106079,0.106079,0.000000,0.106079,0.106079,...,0.119347,0.867386,0.013268,1,1,0,0,1.0,0.0,0.0
3,1,2021,3,0.033165,0.033165,0.033165,0.033165,0.000000,0.033165,0.033165,...,0.047706,0.937752,0.014541,1,0,0,1,0.0,0.0,1.0
4,1,2021,4,0.868348,0.868348,0.868348,0.868348,0.000000,1.128852,1.128852,...,0.880728,0.106892,0.012380,1,1,0,0,1.0,0.0,0.0


In [ ]:
# ============================================================
# 12. Save article-level and quarterly sentiment datasets
# ============================================================

output_file = "finbert_sentiment_features_final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Article Level Sentiment", index=False)
    quarterly_sentiment.to_excel(writer, sheet_name="Quarterly Sentiment", index=False)

print("Saved:", output_file)

Saved: finbert_sentiment_features_final.xlsx


In [ ]:
# ============================================================
# 13. Download file in Google Colab
# ============================================================

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>